<a href="https://colab.research.google.com/github/saurabh2836/ai-ml-engineering-saurabh-kamble/blob/main/Copy_of_Quantization_in_AI_ML_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

What is Quantization in Machine Learning ?

Modern LLMs and deep learning models have grown enormous (billions to trillions of parameters). Storing and running them in FP32 is extremely expensive in terms of:

*   Memory (VRAM/GPU/Edge device RAM)
*   Inference speed/latency
*   Power consumption
*   Cost (cloud bills)
*   Computation Cost












In [ ]:
!pip install -q --upgrade bitsandbytes

In [ ]:
# imports

import os
import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, set_seed
from peft import LoraConfig, PeftModel

In [ ]:
# Constants

BASE_MODEL = "meta-llama/Llama-3.2-3B"

Log in to HuggingFace

If you don't already have a HuggingFace account, visit https://huggingface.co to sign up and create a token.

Then select the Secrets for this Notebook by clicking on the key icon in the left, and add a new secret called HF_TOKEN with the value as your token.

In [ ]:
# Log in to HuggingFace

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

## Trying out different Quantization


# Load the Base Model using 16 bit


In [ ]:
# Load the Base Model without quantization

base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map="auto")

In [ ]:
print(f"Memory footprint: {base_model.get_memory_footprint() / 1e9:,.1f} GB")

In [ ]:
base_model

#Explain the above layers in details
This is the architecture of a Llama-based decoder-only Transformer model (specifically LlamaForCausalLM from Hugging Face Transformers).
It is not the standard Llama 3.1 8B (which uses 4096 hidden size and 32 layers).
This appears to be a smaller/pruned/distilled variant — very likely something like Llama-3.1-Minitron-8B or a similar custom 8B-class model with reduced dimensions (hidden size = 3072 instead of 4096).
High-Level OverviewThis is a Decoder-only Transformer (same family as GPT, Llama, Mistral, Gemma, etc.).
It is designed for autoregressive next-token prediction (causal language modeling).

1. Key Stats (approximate):Hidden dimension
2. (hidden_size): 3072
1. Number of layers: 28
2. Vocabulary size: 128256 (Llama 3 tokenizer)


1.  MLP intermediate size: 8192
2.  Attention type: Grouped-Query Attention (GQA)








##Component-by-Component Breakdown

1. model: Contains all the transformer layers.

2. lm_head: Takes the final hidden states and projects them to vocabulary logits for next-token prediction.

1.   Note: bias=False (common in modern LLMs for efficiency).








##2. Embedding Layer



1.   Turns token IDs (0 to 128255) into dense vectors of size 3072.

2.   This is a learnable lookup table: 128256 × 3072 ≈ 394M parameters.

1.   No positional embedding here — Rotary Embeddings (RoPE) are added later.









##3. LlamaModel (The Core Transformer)


**Contains:embed_tokens**


1.   28 LlamaDecoderLayers (the repeated blocks)


2.   Final norm (RMSNorm)


1.   rotary_emb (shared across layers)










##Restart your session!

In order to load the next model and clear out the cache of the last model, you'll now need to go to Runtime >> Restart session and run the initial cells (imports and HuggingFace login) again.

This is to clean out the GPU.

# Load the Base Model using 8 bit


In [ ]:
# Load the Base Model using 8 bit

quant_config = BitsAndBytesConfig(load_in_8bit=True)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)

In [ ]:
print(f"Memory footprint: {base_model.get_memory_footprint() / 1e9:,.1f} GB")

In [ ]:
base_model

##Restart your session!

In order to load the next model and clear out the cache of the last model, you'll now need to go to Runtime >> Restart session and run the initial cells (imports and HuggingFace login) again.

This is to clean out the GPU.

# Load the Base Model using 4 bit


In [ ]:
# Load the Tokenizer and the Base Model using 4 bit

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)

In [ ]:
print(f"Memory footprint: {base_model.get_memory_footprint() / 1e9:,.1f} GB")

In [ ]:
base_model